# # Robert Bubble Test ｡˚○ ｡˚○ ｡˚○ ｡˚○

<div class="alert alert-block alert-success">

## Part 4: Adding Forcing

TBD
- What is forcing 
- Show how to add forcing to code below
- Have a pre-prepared .py script
- Have user run in terminal
- Explore outputs of the plume

In [ ]:
# Initialize the output before the simulation starts, compute until the desired time 
block.make_outputs(block_vars, current_time)

# Run the simulation in a loop
while not block.intg.stop(block.inc_cycle(), current_time):

    # Each time step of the simulation is determined by the cfl number and the sound speed 
    dt = block.max_time_step(block_vars)

    # Output to let us know the code is running
    block.print_cycle_info(block_vars, current_time, dt)

    # For each cycle, a multi-stage method (here rk3) is used to advance block_vars
    for stage in range(len(block.intg.stages)):
        block.forward(block_vars, dt, stage)

    # Check for any errors
    err = block.check_redo(block_vars)
    if err > 0:
        continue  # redo current step
    if err < 0:
        break  # terminate

    # Progress the time and make outputs 
    current_time += dt
    block.make_outputs(block_vars, current_time)

# Make the final outputs and clean up the internal states in MeshBlock
block.finalize(block_vars, current_time)

<div class="alert alert-info">

## Using Multiple CPU Cores

In order to use more than one CPU core, we will need to add a new dictionary entry to our `.yaml` file: $\texttt{distribute}$

### Define Distribute Properties ($\texttt{distribute}$)

The dictionary entry, $\texttt{distribute}$, describes how parallel computing is accomplished. Copy the following into your `.yaml` file: 

</div>

```yaml
distribute:
  backend: gloo
  layout: slab
  nb2: 4
  nb3: 1
  verbose: false

Which will allow us to use up to 4 CPU cores (nb2*nb3)

<div class="alert alert-info">

## Step-by-step running Robert in terminal


1. Open a terminal and activate your $\texttt{PADDLE}$ conda environment
2. `cd` into the directory storing the `robert.yaml` and `robert.py` (the directory with this notebook in it)
3. Ensure that `robert.yaml` has the distribute text labeled above
4. Run the following code in terminal (4 = number of cores = nb2 * nb1)
    ```
    torchrun --nproc-per-node=4 robert.py
    ```
4. Once finished, you'll notice a bundle of `.nc` files with `.out1` in them. This code combines them into a `main.nc` 
    ```
    pd-combine 1 -o main
    ```
5. Then to view
    ```
    ncview straka-main.nc
    ```

</div>

<div class="alert alert-warning">

Note that in the beta version of $\texttt{PADDLE}$, running things in terminal can sometimes be unstable. 

In particular, if an error occurs, its possible that sockets opened by using multiple cores will not be closed, which can cause a myriad of problems down the line. 

In order to manually close sockets, do the following in a terminal

`lsof -i:29501`
`pkill -9 python`

</div>